# 031 — Seasonal Feature Engineering

Objetivo: predecir la **anomalía media de precipitación de los próximos 3 meses**.

$
y_t = \frac{P'_{t+1}+P'_{t+2}+P'_{t+3}}{3}
$

Se construyen dos experimentos:

- **S1 — ENSO seasonal:** ENSO actual + media móvil de 3 meses + estacionalidad.
- **S2 — ENSO + precipitación reciente:** S1 + anomalía actual + media móvil de anomalía de 3 meses.

La climatología se calcula solo con entrenamiento para evitar leakage.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

TRAIN_END = pd.Timestamp("2010-12-01")
VAL_END = pd.Timestamp("2017-12-01")
TARGET_COLUMN = "target_anomaly_next3m"

print("Proyecto:", PROJECT_ROOT)

Proyecto: /home/gustavo-paredes/Documents/Developer/Yachay/enso-ml


## 1. Cargar ENSO + precipitación

In [15]:
candidate_files = [
    DATA_PROCESSED / "enso_precip.csv",
    DATA_PROCESSED / "enso_precipitation.csv",
]

source_file = next((p for p in candidate_files if p.exists()), None)

if source_file is None:
    raise FileNotFoundError(
        "No encontré enso_precip.csv ni enso_precipitation.csv en data/processed/."
    )

df = pd.read_csv(source_file, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Archivo:", source_file.name)
print("Shape:", df.shape)
print("Fechas:", df["date"].min(), "->", df["date"].max())
df.head()

Archivo: enso_precip.csv
Shape: (912, 12)
Fechas: 1950-01-01 00:00:00 -> 2025-12-01 00:00:00


,date,nino12,nino3,nino34,nino4,soi,tni,mei,oni,precipitation_mm,month,precip_anomaly
0,1950-01-01,-1.20,-1.34,-1.05,-0.69,0.54,0.624,NaN,-1.53,225.615542,1,-100.144358
1,1950-02-01,-1.27,-1.60,-1.50,-1.10,1.58,0.445,NaN,-1.34,295.464787,2,-72.573706
2,1950-03-01,-0.68,-0.96,-1.07,-0.91,1.79,0.382,NaN,-1.16,252.391600,3,-147.498984
3,1950-04-01,-1.22,-0.98,-0.91,-0.76,1.67,0.311,NaN,-1.18,307.984600,4,-39.606670
4,1950-05-01,-0.61,-1.33,-1.30,-0.79,0.55,0.124,NaN,-1.07,144.824903,5,-121.678190


In [16]:
required_columns = [
    "date", "precipitation_mm",
    "nino12", "nino3", "nino34", "nino4", "soi", "tni"
]

missing = [c for c in required_columns if c not in df.columns]
assert not missing, f"Faltan columnas: {missing}"
assert not df["date"].duplicated().any(), "Hay fechas duplicadas."

expected_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="MS")
assert len(expected_dates) == len(df), "La serie no tiene exactamente un registro por mes."
assert np.array_equal(expected_dates.values, df["date"].values), (
    "La serie tiene meses faltantes o fechas fuera de orden."
)

print("Validación temporal: OK")

Validación temporal: OK


## 2. Anomalía de precipitación sin leakage

In [17]:
df["month"] = df["date"].dt.month

train_climatology = (
    df.loc[df["date"] <= TRAIN_END]
      .groupby("month")["precipitation_mm"]
      .mean()
)

assert len(train_climatology) == 12

df["climatology_mm"] = df["month"].map(train_climatology)
df["precip_anomaly"] = df["precipitation_mm"] - df["climatology_mm"]

train_climatology

month
1     324.663879
2     370.387311
3     398.849610
4     346.349142
5     266.107109
6     187.664631
7     158.699567
8     137.969504
9     154.659881
10    171.603695
11    163.513859
12    232.210664
Name: precipitation_mm, dtype: float64

## 3. Target estacional de los próximos 3 meses

In [18]:
future_anomalies = pd.concat(
    [
        df["precip_anomaly"].shift(-1),
        df["precip_anomaly"].shift(-2),
        df["precip_anomaly"].shift(-3),
    ],
    axis=1,
)

df[TARGET_COLUMN] = future_anomalies.mean(axis=1, skipna=False)

df["target_start"] = df["date"] + pd.DateOffset(months=1)
df["target_center"] = df["date"] + pd.DateOffset(months=2)
df["target_end"] = df["date"] + pd.DateOffset(months=3)

df[
    ["date", "precip_anomaly", "target_start", "target_center", "target_end", TARGET_COLUMN]
].head(8)

,date,precip_anomaly,target_start,target_center,target_end,target_anomaly_next3m
0,1950-01-01,-99.048337,1950-02-01,1950-03-01,1950-04-01,-86.581692
1,1950-02-01,-74.922525,1950-03-01,1950-04-01,1950-05-01,-102.034920
2,1950-03-01,-146.458010,1950-04-01,1950-05-01,1950-06-01,-62.551056
3,1950-04-01,-38.364542,1950-05-01,1950-06-01,1950-07-01,-47.238048
4,1950-05-01,-121.282206,1950-06-01,1950-07-01,1950-08-01,-4.635668
5,1950-06-01,-28.006418,1950-07-01,1950-08-01,1950-09-01,1.857987
6,1950-07-01,7.574480,1950-08-01,1950-09-01,1950-10-01,-21.370414
7,1950-08-01,6.524933,1950-09-01,1950-10-01,1950-11-01,-51.418437


## 4. Estacionalidad del mes central de la ventana objetivo

In [19]:
df["target_month"] = df["target_center"].dt.month

df["target_month_sin"] = np.sin(2 * np.pi * df["target_month"] / 12)
df["target_month_cos"] = np.cos(2 * np.pi * df["target_month"] / 12)

## 5. Features ENSO compactas

In [20]:
enso_variables = ["nino12", "nino3", "nino34", "nino4", "soi", "tni"]

for variable in enso_variables:
    df[f"{variable}_ma3"] = (
        df[variable]
        .rolling(window=3, min_periods=3)
        .mean()
    )

enso_features = []
for variable in enso_variables:
    enso_features.extend([variable, f"{variable}_ma3"])

seasonal_features = ["target_month_sin", "target_month_cos"]

print("ENSO features:", len(enso_features))
print(enso_features)

ENSO features: 12
['nino12', 'nino12_ma3', 'nino3', 'nino3_ma3', 'nino34', 'nino34_ma3', 'nino4', 'nino4_ma3', 'soi', 'soi_ma3', 'tni', 'tni_ma3']


## 6. Memoria reciente de precipitación

In [21]:
df["precip_anomaly_ma3"] = (
    df["precip_anomaly"]
    .rolling(window=3, min_periods=3)
    .mean()
)

precip_features = [
    "precip_anomaly",
    "precip_anomaly_ma3",
]

## 7. Definir S1 y S2

In [22]:
feature_columns_s1 = enso_features + seasonal_features
feature_columns_s2 = feature_columns_s1 + precip_features

FEATURE_SETS = {
    "S1_ENSO_SEASONAL": feature_columns_s1,
    "S2_ENSO_PRECIP_SEASONAL": feature_columns_s2,
}

for name, features in FEATURE_SETS.items():
    print(name, "->", len(features), "features")

assert len(feature_columns_s1) == 14
assert len(feature_columns_s2) == 16

S1_ENSO_SEASONAL -> 14 features
S2_ENSO_PRECIP_SEASONAL -> 16 features


In [23]:
forbidden = {
    "date", "target_start", "target_center", "target_end",
    TARGET_COLUMN, "climatology_mm", "target_month"
}

for name, features in FEATURE_SETS.items():
    assert len(features) == len(set(features)), f"{name} tiene features duplicadas."
    assert not (forbidden & set(features)), (
        f"{name} contiene columnas prohibidas: {forbidden & set(features)}"
    )

print("Listas de features: OK")

Listas de features: OK


## 8. Construir datasets supervisados

In [24]:
METADATA_COLUMNS = [
    "date",
    "target_start",
    "target_center",
    "target_end",
    "precip_anomaly",
    "precip_anomaly_ma3",
]

def build_model_dataset(dataframe, features):
    cols = METADATA_COLUMNS + features + [TARGET_COLUMN]
    cols = list(dict.fromkeys(cols))
    return dataframe[cols].dropna().reset_index(drop=True)

model_s1 = build_model_dataset(df, feature_columns_s1)
model_s2 = build_model_dataset(df, feature_columns_s2)

DATASETS = {
    "S1_ENSO_SEASONAL": model_s1,
    "S2_ENSO_PRECIP_SEASONAL": model_s2,
}

for name, data in DATASETS.items():
    print(
        name,
        "| shape:", data.shape,
        "| date:", data["date"].min().date(), "->", data["date"].max().date(),
        "| target:", data["target_start"].min().date(), "->", data["target_end"].max().date(),
    )

S1_ENSO_SEASONAL | shape: (892, 21) | date: 1950-03-01 -> 2025-09-01 | target: 1950-04-01 -> 2025-12-01
S2_ENSO_PRECIP_SEASONAL | shape: (892, 21) | date: 1950-03-01 -> 2025-09-01 | target: 1950-04-01 -> 2025-12-01


## 9. Comprobar splits sin ventanas cruzadas

In [25]:
def split_counts(data):
    train = data[data["target_end"] <= TRAIN_END]

    val = data[
        (data["target_start"] >= pd.Timestamp("2011-01-01"))
        & (data["target_end"] <= VAL_END)
    ]

    test = data[
        data["target_start"] >= pd.Timestamp("2018-01-01")
    ]

    return len(train), len(val), len(test)

for name, data in DATASETS.items():
    print(name, "-> Train / Val / Test:", split_counts(data))

S1_ENSO_SEASONAL -> Train / Val / Test: (715, 82, 91)
S2_ENSO_PRECIP_SEASONAL -> Train / Val / Test: (715, 82, 91)


## 10. Distribución del target

In [26]:
model_s1[TARGET_COLUMN].describe()

count    892.000000
mean      -0.883647
std       68.937011
min     -182.369581
25%      -38.346555
50%      -12.065012
75%       22.542505
max      390.986055
Name: target_anomaly_next3m, dtype: float64

## 11. Guardar datasets

In [27]:
OUTPUT_FILES = {
    "S1_ENSO_SEASONAL": DATA_PROCESSED / "seasonal_model_s1.csv",
    "S2_ENSO_PRECIP_SEASONAL": DATA_PROCESSED / "seasonal_model_s2.csv",
}

for name, data in DATASETS.items():
    data.to_csv(OUTPUT_FILES[name], index=False)
    print(name, "->", OUTPUT_FILES[name].name, data.shape)

S1_ENSO_SEASONAL -> seasonal_model_s1.csv (892, 21)
S2_ENSO_PRECIP_SEASONAL -> seasonal_model_s2.csv (892, 21)


## Resultado

Se generan:

- `seasonal_model_s1.csv`
- `seasonal_model_s2.csv`

El test no se utiliza aquí; queda reservado para la etapa final.